# TP15 - Méthodes d'ensemble

**Objectif :** pousser la performance proprement et savoir dire ce que le modèle rate.

## 1. Remettre le décor en place

Chargement des données

In [1]:
import pandas as pd

df = pd.read_parquet("../data/ml/nutriscope_ml_sample.parquet")
display(df.shape)
display(df["split"].value_counts())
display(df["nutriscore_grade"].value_counts(normalize=True))
display(df)

(30041, 21)

split
train    24076
test      5965
Name: count, dtype: int64

nutriscore_grade
e    0.286409
d    0.264372
c    0.207550
a    0.134150
b    0.107520
Name: proportion, dtype: float64

,code,product_name,brands,pnns_groups_1,pnns_groups_2,nova_group,additives_n,energy-kcal_100g,fat_100g,saturated-fat_100g,...,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,nutriscore_score,nutriscore_grade,completeness,last_modified,split
0,0000112302660,Soupe potiron et kiri,Liebig,Fruits and vegetables,Soups,4,4,38.300000,1.50000,1.000000,...,2.000000,0.800000,0.90000,0.740000,34.312500,3,c,0.6875,2026-03-14,train
1,0000112302705,velouté de tomates et pointe d'origan,Liebig,Fruits and vegetables,Soups,4,0,28.000000,0.50000,0.300000,...,2.400000,1.000000,0.70000,0.670000,72.977007,1,b,0.7625,2026-03-19,test
2,0000112302754,pur soup' mouliné de 5 légumes verts,liebig,Composite foods,One-dish meals,3,0,24.802376,0.16035,0.044483,...,0.738538,0.837462,0.81314,0.035146,23.242188,0,a,0.9750,2026-03-26,train
3,0000112307261,Velouté de 10 légumes,Liebig,Fruits and vegetables,Soups,4,0,43.000000,1.60000,0.600000,...,1.300000,0.800000,0.70000,0.670000,34.981336,3,c,0.9000,2026-03-26,train
4,0000300000004,Lentilles Brunes Bio,Kazidomi,Composite foods,One-dish meals,3,0,54.000000,0.30000,0.000000,...,0.000000,4.000000,4.00000,0.080000,100.000000,-7,a,0.8875,2026-03-17,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30036,9950601010759,Graine de lin brun,NaN,Cereals and potatoes,Cereals,<NA>,<NA>,527.000000,43.00000,4.300000,...,0.900000,NaN,20.10000,0.120000,NaN,-5,b,0.3625,2026-03-24,test
30037,99623456,Petales de patates douces,Croustisud,Salty snacks,Appetizers,3,0,538.000000,35.10000,4.000000,...,27.000000,1.865500,4.30000,0.300000,0.000000,17,d,0.5750,2026-03-20,train
30038,99895488,Langouste cuite coupée en 2,NaN,Fish Meat Eggs,Fish and seafood,<NA>,<NA>,136.000000,1.94000,0.297000,...,0.000000,NaN,26.40000,0.500000,NaN,-4,a,0.6750,2026-03-20,train
30039,99993306,Pizza thon cuite sur pierre,Trattoria Alfredo,Composite foods,Pizza pies and quiches,<NA>,<NA>,231.000000,9.80000,3.500000,...,5.500000,NaN,10.00000,1.230000,NaN,12,d,0.4750,2026-03-20,test


Choix des colonnes

In [2]:
NUTRIMENTS = [
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g", "carbohydrates_100g",
    "sugars_100g", "proteins_100g", "salt_100g"
]
OPTIONNELLES = [
    "fiber_100g", "fruits-vegetables-legumes_100g", "nova_group", "additives_n"
]
CATEGORIELLES = ["pnns_groups_1", "pnns_groups_2"]

FEATURES = NUTRIMENTS + OPTIONNELLES + CATEGORIELLES

FORBIDDEN = ["nutriscore_score", "code", "product_name", "brands", "split"]

for col in FORBIDDEN:
    assert col not in FEATURES, f"{col} ne doit pas être une feature"

Préservation du train

In [3]:
train = df[df["split"] == "train"]

X_train = train[FEATURES]
y_train = train["nutriscore_grade"]

Définition des règles

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = ["f1_macro", "accuracy", "balanced_accuracy"]

def evaluate_cv(name: str, model, X: pd.DataFrame) -> dict:
    """Évalue un modèle par validation croisée sur le train, avec les plis et métriques communs.

    Tous les modèles passent par cette fonction : ils sont donc comparés sur les
    mêmes plis (`cv`) et les mêmes métriques (`SCORING`), condition d'un tableau
    comparatif valable.

    Args:
        name: nom du modèle, affiché dans le tableau des résultats.
        model: modèle ou Pipeline scikit-learn, non entraîné.
        X: features du train (la cible utilisée est `y_train`).

    Returns:
        Dictionnaire avec la moyenne des scores sur les 5 plis, l'écart-type du
        F1 macro, et les temps moyens d'ajustement et de prédiction (en secondes).
    """
    r = cross_validate(model, X, y_train, cv=cv, scoring=SCORING)
    return {
        "model": name,
        "f1_macro": r["test_f1_macro"].mean(),
        "f1_std": r["test_f1_macro"].std(),
        "accuracy": r["test_accuracy"].mean(),
        "balanced_accuracy": r["test_balanced_accuracy"].mean(),
        "fit_time": r["fit_time"].mean(), "score_time": r["score_time"].mean()
    }

results = []


Préparation et entraînement

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

import numpy as np

def add_derived_features(X: pd.DataFrame) -> pd.DataFrame:
    """Ajoute deux ratios et l'écart d'énergie (règle 4/4/9 du TP 9)."""
    X = X.copy()
    X["sat_fat_ratio"] = (X["saturated-fat_100g"] / X["fat_100g"].replace(0, np.nan)).fillna(0)
    X["sugar_ratio"] = (X["sugars_100g"] / X["carbohydrates_100g"].replace(0, np.nan)).fillna(0)
    energy_calc = 4 * X["carbohydrates_100g"] + 4 * X["proteins_100g"] + 9 * X["fat_100g"]
    X["energy_gap"] = X["energy-kcal_100g"] - energy_calc
    return X

DERIVEES = ["sat_fat_ratio", "sugar_ratio", "energy_gap"]


optional_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("nutr", StandardScaler(), NUTRIMENTS + DERIVEES),
    ("opt", optional_pipeline, OPTIONNELLES),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=80), CATEGORIELLES)
])

prep_pipeline = Pipeline([
    ("derived", FunctionTransformer(add_derived_features)),
    ("prep", preprocessor),
])

(24076, 66)


In [6]:
from sklearn.linear_model import LogisticRegression


logreg = Pipeline([("prep", prep_pipeline), ("model", LogisticRegression(max_iter=3000))])
results.append(evaluate_cv("logreg", logreg, X_train))

display(pd.DataFrame(results).set_index("model").round(3))

,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.716,0.037


In [7]:
from sklearn.ensemble import RandomForestClassifier


rnd_forest = Pipeline([("prep", prep_pipeline), ("model", RandomForestClassifier(random_state=42))])
results.append(evaluate_cv("rnd_forest", rnd_forest, X_train))

display(pd.DataFrame(results).set_index("model").round(3))

,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.716,0.037
rnd_forest,0.858,0.007,0.877,0.854,1.936,0.095
